# PDP SCALING ANALYSIS & BATCH SOLVER NOTEBOOK

Sổ tay này được thiết kế để phục vụ việc **chạy độc lập từng thuật toán** trên các cỡ dữ liệu khác nhau, lưu trữ kết quả riêng biệt cho từng tệp tin và tự động vẽ biểu đồ đường so sánh hiệu năng theo quy mô dữ liệu tăng dần (Scaling Analysis).

In [1]:
# 1. Khởi tạo môi trường và Import thư viện
%matplotlib inline
import sys
import os
import time
import re
import matplotlib.pyplot as plt

# Thêm thư mục cha vào sys.path để import các module từ src/
sys.path.append(os.path.abspath('..'))

from src.utils.parser import PDPParser
from src.utils.visualizer import PDPVisualizer
from src.algorithms.greedy_insertion import GreedyPairInsertionSolver
from src.algorithms.local_search import LocalSearchSolver
from src.algorithms.alns import ALNSSolver
from src.algorithms.milp import MILPPDPSolver
from src.models.solution import PDPSolution

print('Đã thiết lập môi trường thành công!')

Đã thiết lập môi trường thành công!


# PHẦN 1: BỘ CHẠY HÀNG LOẠT ĐỘC LẬP CHO TỪNG THUẬT TOÁN
Hàm dưới đây giúp bạn lựa chọn chạy **duy nhất 1 thuật toán** trên toàn bộ các tệp của một thư mục dữ liệu cụ thể và lưu kết quả vào thư mục tương ứng.

In [2]:
def save_result_to_file(size_name, algo_name, filename, solution, elapsed_time):
    '''
    Tự động lưu báo cáo chi tiết lộ trình chạy của từng tệp ra đĩa.
    Đường dẫn: ../results/{cỡ}/{thuật_toán}/{tên_file}.txt
    '''
    algo_dir = os.path.join('..', 'results', size_name, algo_name)
    os.makedirs(algo_dir, exist_ok=True)
    
    output_path = os.path.join(algo_dir, filename)
    is_feasible, errors = solution.check_feasibility()
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('=' * 60 + '\n')
        f.write(f'        BÁO CÁO KẾT QUẢ CHẠY CHI TIẾT - TỆP: {filename}\n')
        f.write('=' * 60 + '\n\n')
        f.write(f'Tệp dữ liệu gốc      : {filename}\n')
        f.write(f'Cỡ dữ liệu           : {size_name}\n')
        f.write(f'Giải thuật sử dụng   : {algo_name}\n')
        f.write(f'Trạng thái khả thi   : {"HỢP LỆ" if is_feasible else "KHÔNG HỢP LỆ"}\n')
        f.write(f'Tổng quãng đường di chuyển: {solution.calculate_total_cost():.2f}\n')
        f.write(f'Thời gian tính toán  : {elapsed_time:.4f} giây\n\n')
        
        if not is_feasible:
            f.write('DANH SÁCH LỖI VI PHẠM RÀNG BUỘC:\n')
            for i, err in enumerate(errors, 1):
                f.write(f'  {i}. {err}\n')
            f.write('\n')
            
        f.write('LỘ TRÌNH XE CHI TIẾT:\n')
        f.write('-' * 60 + '\n')
        for vehicle_id, route in solution.routes.items():
            f.write(f'Xe {vehicle_id} : {route}\n')
        f.write('-' * 60 + '\n')

def run_batch_for_algorithm(data_folder, algo_name, max_files=5, milp_time_limit=5, alns_iterations=10):
    '''
    Quét thư mục và chạy độc lập DUY NHẤT 1 thuật toán được chỉ định.
    '''
    clean_path = data_folder.replace('\\', '/').rstrip('/')
    parts = clean_path.split('/')
    size_name = 'pdp_unknown'
    for part in reversed(parts):
        if 'pdp' in part:
            size_name = part
            break
    else:
        size_name = parts[-1] if parts else 'pdp_unknown'

    if not os.path.exists(data_folder):
        print(f'[Lỗi] Không tìm thấy thư mục: {data_folder}')
        return

    files = [f for f in os.listdir(data_folder) if f.endswith('.txt')]
    files.sort()
    selected_files = files[:max_files]
    
    print(f'=== CHẠY ĐỘC LẬP ALGO: {algo_name} | CỠ: {size_name} ===')
    print(f'-> Quét thấy {len(files)} file. Tiến hành chạy {len(selected_files)} file...')

    costs = []
    times = []

    for idx, fname in enumerate(selected_files, 1):
        file_path = os.path.join(data_folder, fname)
        try:
            instance = PDPParser.parse_li_lim_format(file_path)
        except Exception as e:
            print(f'  [{idx}] Lỗi đọc file {fname}: {e}')
            continue
            
        t_start = time.time()
        
        # Chỉ khởi chạy duy nhất 1 thuật toán được chọn
        if algo_name == 'Greedy':
            sol = GreedyPairInsertionSolver(instance).solve()
        elif algo_name == 'LocalSearch':
            # Tạo lời giải ban đầu bằng Greedy trước làm đầu vào cho Local Search
            greedy_sol = GreedyPairInsertionSolver(instance).solve()
            sol = LocalSearchSolver(instance, initial_solution=greedy_sol).solve()
        elif algo_name == 'ALNS':
            greedy_sol = GreedyPairInsertionSolver(instance).solve()
            sol = ALNSSolver(instance, initial_solution=greedy_sol, iterations=alns_iterations).solve()
        elif algo_name == 'MILP':
            sol = MILPPDPSolver(instance, time_limit=milp_time_limit).solve()
        else:
            print(f'[Error] Thuật toán không hợp lệ: {algo_name}')
            return
            
        dt = time.time() - t_start
        cost = sol.calculate_total_cost()
        
        costs.append(cost)
        times.append(dt)
        
        # Lưu kết quả chi tiết từng tệp
        save_result_to_file(size_name, algo_name, fname, sol, dt)
        print(f'  [{idx}/{len(selected_files)}] File {fname} -> Cost: {cost:.1f} | Time: {dt:.4f}s')
        
    avg_c = sum(costs) / len(costs) if costs else 0
    avg_t = sum(times) / len(times) if times else 0
    print(f'==> KẾT QUẢ TRUNG BÌNH NHÓM: Quãng đường = {avg_c:.2f} | Thời gian = {avg_t:.4f}s\n')

### Ví dụ Khởi chạy Độc lập:
Bạn có thể chạy thử nghiệm thuật toán của mình độc lập bằng các ô code dưới đây. Chỉ cần đổi đường dẫn và chọn số lượng tệp muốn chạy thử (`max_files`).

In [3]:
# Chạy hàng loạt cho riêng thuật toán Greedy cỡ 100
run_batch_for_algorithm(data_folder='../data/pdp_100/pdp_100', algo_name='Greedy', max_files=5)

=== CHẠY ĐỘC LẬP ALGO: Greedy | CỠ: pdp_100 ===
-> Quét thấy 56 file. Tiến hành chạy 5 file...
[Parser] Đang đọc file dữ liệu PDP: ../data/pdp_100/pdp_100\lc101.txt
  [1/5] File lc101.txt -> Cost: 1518.1 | Time: 0.0190s
[Parser] Đang đọc file dữ liệu PDP: ../data/pdp_100/pdp_100\lc102.txt
  [2/5] File lc102.txt -> Cost: 1498.7 | Time: 0.0176s
[Parser] Đang đọc file dữ liệu PDP: ../data/pdp_100/pdp_100\lc103.txt
  [3/5] File lc103.txt -> Cost: 1539.6 | Time: 0.0163s
[Parser] Đang đọc file dữ liệu PDP: ../data/pdp_100/pdp_100\lc104.txt
  [4/5] File lc104.txt -> Cost: 1446.8 | Time: 0.0162s
[Parser] Đang đọc file dữ liệu PDP: ../data/pdp_100/pdp_100\lc105.txt
  [5/5] File lc105.txt -> Cost: 1494.2 | Time: 0.0160s
==> KẾT QUẢ TRUNG BÌNH NHÓM: Quãng đường = 1499.49 | Thời gian = 0.0170s



In [ ]:
# Chạy hàng loạt cho riêng thuật toán LocalSearch cỡ 100
run_batch_for_algorithm(data_folder='../data/pdp_100/pdp_100', algo_name='LocalSearch', max_files=5)

In [ ]:
# Chạy hàng loạt cho riêng thuật toán ALNS cỡ 100
run_batch_for_algorithm(data_folder='../data/pdp_100/pdp_100', algo_name='ALNS', max_files=5, alns_iterations=10)

In [ ]:
# Chạy hàng loạt cho riêng thuật toán MILP cỡ 100
run_batch_for_algorithm(data_folder='../data/pdp_100/pdp_100', algo_name='MILP', max_files=5, milp_time_limit=3)

# PHẦN 2: QUÉT KẾT QUẢ TỪ ĐĨA & VẼ BIỂU ĐỒ ĐƯỜNG TĂNG TRƯỞNG
Sau khi đã thu thập đủ kết quả chạy của các cỡ dữ liệu (ví dụ cỡ 100, 200, 400, 600...) trên đĩa, chạy ô code dưới đây sẽ tự động quét thư mục `results/`, đọc báo cáo từ các file `.txt` để phân tích và vẽ biểu đồ đường so sánh.

In [ ]:
def parse_saved_results(results_dir='../results'):
    '''
    Tự động đọc và phân tích toàn bộ tệp .txt trong results/ để trích xuất số liệu trung bình.
    '''
    if not os.path.exists(results_dir):
        print(f'[Error] Chưa tồn tại thư mục kết quả: {results_dir}')
        return {}

    # Cấu trúc: data_summary[cỡ_dữ_liệu][thuật_toán] = {'costs': [], 'times': []}
    data_summary = {}

    # Duyệt qua các thư mục cỡ dữ liệu (ví dụ: pdp_100, pdp_200...)
    for size_folder in os.listdir(results_dir):
        size_path = os.path.join(results_dir, size_folder)
        if not os.path.isdir(size_path):
            continue
            
        data_summary[size_folder] = {}
        
        # Duyệt qua các thư mục thuật toán (Greedy, ALNS...)
        for algo_folder in os.listdir(size_path):
            algo_path = os.path.join(size_path, algo_folder)
            if not os.path.isdir(algo_path):
                continue
                
            costs = []
            times = []
            
            # Đọc từng tệp tin kết quả
            for filename in os.listdir(algo_path):
                if not filename.endswith('.txt'):
                    continue
                file_path = os.path.join(algo_path, filename)
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    
                    # Dùng Regex để tìm giá trị quãng đường và thời gian
                    cost_match = re.search(r'Tổng\s+(?:chi\s+phí|quãng\s+đường)\s+.*?:\s*([\d.]+)', content, re.IGNORECASE)
                    time_match = re.search(r'Thời\s+gian\s+tính\s+toán\s*:\s*([\d.]+)', content, re.IGNORECASE)
                    
                    if cost_match:
                        costs.append(float(cost_match.group(1)))
                    if time_match:
                        times.append(float(time_match.group(1)))
                except Exception:
                    continue
            
            if costs:
                data_summary[size_folder][algo_folder] = {
                    'avg_cost': sum(costs) / len(costs),
                    'avg_time': sum(times) / len(times)
                }
                
    return data_summary

In [ ]:
def plot_scaling_curves():
    '''
    Quét dữ liệu thực tế và vẽ hai biểu đồ dạng đường thể hiện tính tăng trưởng quy mô (Complexity Curves).
    '''
    data = parse_saved_results()
    if not data:
        print('Không tìm thấy dữ liệu kết quả để vẽ biểu đồ!')
        return

    # Hàm trích xuất phần số từ tên thư mục để làm nhãn trục X và sắp xếp tăng dần
    def extract_size_number(name):
        num_match = re.search(r'\d+', name)
        return int(num_match.group(0)) if num_match else 0

    # Lấy danh sách các cỡ dữ liệu hiện có và sắp xếp tăng dần
    sorted_sizes = sorted(list(data.keys()), key=extract_size_number)
    x_labels = [str(extract_size_number(s)) for s in sorted_sizes]
    x_values = [extract_size_number(s) for s in sorted_sizes]
    
    if not sorted_sizes:
        print('Chưa có kết quả chạy của cỡ dữ liệu nào trên đĩa!')
        return

    # Lấy danh sách toàn bộ thuật toán
    algorithms = ['Greedy', 'LocalSearch', 'ALNS', 'MILP']
    
    # Chuẩn bị cấu trúc vẽ đường
    # plot_data[algo] = {'x': [], 'cost': [], 'time': []}
    plot_data = {algo: {'x': [], 'cost': [], 'time': []} for algo in algorithms}

    for idx, size_folder in enumerate(sorted_sizes):
        size_val = x_values[idx]
        for algo in algorithms:
            if algo in data[size_folder]:
                plot_data[algo]['x'].append(size_val)
                plot_data[algo]['cost'].append(data[size_folder][algo]['avg_cost'])
                plot_data[algo]['time'].append(data[size_folder][algo]['avg_time'])
    
    # Tạo biểu đồ
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))
    
    colors = {
        'Greedy': '#1f77b4',       # Xanh dương
        'LocalSearch': '#ff7f0e',  # Cam
        'ALNS': '#2ca02c',         # Xanh lá
        'MILP': '#d62728'          # Đỏ
    }
    markers = {
        'Greedy': 'o',
        'LocalSearch': 's',
        'ALNS': '^',
        'MILP': 'd'
    }
    
    # 1. Biểu đồ Thời gian Trung bình
    for algo in algorithms:
        if plot_data[algo]['x']:
            ax1.plot(plot_data[algo]['x'], plot_data[algo]['time'], 
                     label=algo, color=colors[algo], marker=markers[algo], 
                     linewidth=2.5, markersize=8, linestyle='-')
            
    ax1.set_title('PHÂN TÍCH QUY MÔ: THỜI GIAN THỰC THI TRUNG BÌNH', fontsize=12, weight='bold', pad=12)
    ax1.set_xlabel('Kích cỡ dữ liệu (Số lượng khách hàng)', fontsize=11)
    ax1.set_ylabel('Thời gian chạy trung bình (Giây)', fontsize=11)
    ax1.set_xticks(x_values)
    ax1.set_xticklabels(x_labels)
    ax1.grid(True, linestyle='--', alpha=0.4)
    ax1.legend(title='Giải thuật', loc='upper left', frameon=True, shadow=True)
    
    # 2. Biểu đồ Quãng đường Trung bình (Chất lượng lời giải)
    for algo in algorithms:
        if plot_data[algo]['x']:
            # Bỏ qua MILP nếu không có kết quả thực sự cho các cỡ lớn hơn
            if algo == 'MILP' and all(c == 0 for c in plot_data[algo]['cost']):
                continue
            ax2.plot(plot_data[algo]['x'], plot_data[algo]['cost'], 
                     label=algo, color=colors[algo], marker=markers[algo], 
                     linewidth=2.5, markersize=8, linestyle='-')
            
    ax2.set_title('PHÂN TÍCH QUY MÔ: TỔNG QUÃNG ĐƯỜNG TRUNG BÌNH', fontsize=12, weight='bold', pad=12)
    ax2.set_xlabel('Kích cỡ dữ liệu (Số lượng khách hàng)', fontsize=11)
    ax2.set_ylabel('Quãng đường di chuyển trung bình (m)', fontsize=11)
    ax2.set_xticks(x_values)
    ax2.set_xticklabels(x_labels)
    ax2.grid(True, linestyle='--', alpha=0.4)
    ax2.legend(title='Giải thuật', loc='upper left', frameon=True, shadow=True)
    
    plt.tight_layout()
    plt.show()

# Gọi hàm vẽ biểu đồ
plot_scaling_curves()